In [222]:
import pandas as pd
import numpy as np
import random
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [223]:
SEED = 42 
random.seed(SEED) 
np.random.seed(SEED) 
os.environ["PYTHONHASHSEED"] = str(SEED) 


In [224]:
df = pd.read_csv('../data/data_raw.csv')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118382 entries, 0 to 118381
Data columns (total 75 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   id                                    118382 non-null  object 
 1   description                           113990 non-null  object 
 2   ratings_average                       76428 non-null   object 
 3   ratings_count                         76428 non-null   float64
 4   ratings_recommend_percentage          76428 non-null   float64
 5   price_currency                        118382 non-null  object 
 6   price                                 118382 non-null  float64
 7   price_tax_deductible                  118382 non-null  bool   
 8   price_negotiable                      118382 non-null  bool   
 9   price_net                             34016 non-null   float64
 10  price_vat_rate                        30379 non-null   float64
 11  

In [225]:
#data cleaning and redundant values handling
def clean_raw_data(df):
    df = df.copy()

    df.drop_duplicates(inplace=True)

    cols_with_units = ['weight_kg','mileage_km']
    
    for col in cols_with_units:
        if col in df.columns:
            df[col] = (df[col].astype(str).str.replace(r'[^0-9.]','',regex=True).replace('',np.nan).astype('float64'))
    
    #repeating, strongly_correlated or introducing data leakage columns (based on eda)        
    cols_to_drop = ['mileage_km_raw', 'power_hp', 'price_net', 'price_vat_rate', 'electric_range_city_km']

    df.drop(columns=cols_to_drop,inplace=True)
    return df

In [226]:
#early feature engineering / missing_value imputation from identifiers

def early_feature_engineering(df):
    df = df.copy()
    
    equipment_cols = ['equipment_comfort', 'equipment_entertainment', 'equipment_extra', 'equipment_safety']
    
    # car age from registration
    current_year = 2026
    df['registration_year'] = pd.to_datetime(df['registration_date'], errors='coerce').dt.year
    df['production_year'] = df['production_year'].fillna(df['registration_year'])
    df['car_age'] = current_year - df['production_year']
    
    cols_to_drop = ['production_year', 'registration_date', 'registration_year']
    df.drop(columns=cols_to_drop,inplace=True)
    
    #feature engineering-ratings - Bayesian avarage
    df['ratings_average'] = (df['ratings_average'].astype(str).str.replace(',', '.', regex=False).str.extract(r'(\d+\.?\d*)')[0].astype(float))

    df['ratings_average'] = df['ratings_average'].fillna(2.5)
    df['ratings_count'] = df['ratings_count'].fillna(0)
    df['ratings_recommend_percentage'] = df['ratings_recommend_percentage'].fillna(0)
    
    C = 2.5
    m = 10
        
    df['offer_score'] = (C * m + df['ratings_average'] * df['ratings_count']) / (m + df['ratings_count'])
        
        
    mask_high_recommend = df['ratings_recommend_percentage'] > 85
    df.loc[mask_high_recommend, 'offer_score'] += 0.5
    df['offer_score'] = df['offer_score'].clip(upper=5.0)
        
    ratings_drop = ['ratings_average', 'ratings_count', 'ratings_recommend_percentage']
    df.drop(columns=ratings_drop,inplace=True)

    return df


In [227]:
#missing and unique values handling 
def handle_missing_and_unique(df, col_threshold=80, row_threshold=2,cardinality_threshold=100):
    df = df.copy()
    missing_percent = (df.isnull().sum() / len(df)) * 100
    missing_data = missing_percent[missing_percent > 0]
    
    cols_to_drop = missing_data[missing_data > col_threshold].index.tolist()
    df.drop(columns=cols_to_drop,inplace=True)
    
    cols_to_trim = missing_data[missing_data < row_threshold].index.tolist()
    df.dropna(subset=cols_to_trim,inplace=True)
    
    constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
    df.drop(columns=constant_cols,inplace=True)
    
    object_cols = df.select_dtypes(include=['object']).columns
    high_card_cols = [c for c in object_cols if df[c].nunique() > cardinality_threshold]
    df.drop(columns=high_card_cols, inplace=True)
    
    location_cols= ['latitude','longitude']
    df.drop(columns=location_cols, inplace=True)
    return df



In [228]:
#outlier handling
def handle_outliers(df):
    Q1 = df['price'].quantile(0.25)
    Q3 = df['price'].quantile(0.75)
    IQR = Q3 - Q1
        
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)].copy()
    
    
    numeric_cols = df.select_dtypes(include=['float','int']).columns
    
    for col in numeric_cols:
        lower_limit = df[col].quantile(0.01)
        upper_limit = df[col].quantile(0.99)
            
        df[col] = df[col].clip(lower=lower_limit, upper=upper_limit)
     
    return df



In [229]:
#data filling, encoding, scaling and train-test split
def encode_and_split(df):
    df = df.copy()

    X = df.drop(columns=['price'])
    y = df['price']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    
    zero_fill_cols = ['gears', 'nr_prev_owners']
    for col in zero_fill_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col].fillna(0)
            X_test[col] = X_test[col].fillna(0)


    numeric_cols = X_train.select_dtypes(include=['int','float']).columns
    train_medians = X_train[numeric_cols].median()
    
    X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
    X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)


    object_cols = X_train.select_dtypes(include=['object']).columns
    for col in object_cols:
        if not X_train[col].mode().empty:
            train_mode = X_train[col].mode()[0]        
            X_train[col] = X_train[col].fillna(train_mode)
            X_test[col] = X_test[col].fillna(train_mode)
        else:
            X_train[col] = X_train[col].fillna("Unknown")
            X_test[col] = X_test[col].fillna("Unknown")


    X_train = pd.get_dummies(X_train, drop_first=True)
    X_test = pd.get_dummies(X_test, drop_first=True)
    
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    scaler = StandardScaler()
    
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
    
    return X_train, X_test, y_train, y_test


In [230]:
df_clean = clean_raw_data(pd.read_csv('../data/data_raw.csv'))
df_eng = early_feature_engineering(df_clean)
df_structured = handle_missing_and_unique(df_eng, col_threshold=60, row_threshold=2, cardinality_threshold=100)
df_outliers_handled = handle_outliers(df_structured)
X_train, X_test, y_train, y_test = encode_and_split(df_outliers_handled)

X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

